# Image Processing Pipeline for Genipa Seed Digital Phenotyping

This notebook  organizes the image-processing workflow used to crop tray images, detect the calibration square, segment and label seeds, export seed morphometric measurements, crop individual seeds, and optionally convert the blue EVA background to black.

The notebook is intentionally written as a reproducible pipeline: configuration is centralized, functions are reusable, and execution cells are separated from function definitions.

## 1. Imports and Configuration

Adjust only the paths in `ImageProcessingConfig` before running the pipeline. The default layout assumes that raw images are kept outside version control and that derived outputs are written under `outputs/` or `data/processed/`.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re
import shutil
from typing import Iterable

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from sklearn.cluster import KMeans


@dataclass(frozen=True)
class ImageProcessingConfig:
    # Project paths
    project_root: Path
    original_images_dir: Path
    rectified_images_dir: Path
    annotated_images_dir: Path
    individual_seeds_dir: Path
    black_background_dir: Path
    processed_dataset_path: Path
    germination_metadata_path: Path | None = None

    # Blue EVA color interval in HSV
    lower_blue: tuple[int, int, int] = (90, 50, 50)
    upper_blue: tuple[int, int, int] = (130, 255, 255)

    # Morphology and segmentation parameters
    kernel_size: int = 5
    min_area_ratio: float = 0.00021
    max_area_ratio: float = 0.01
    min_width_ratio: float = 0.01
    max_width_ratio: float = 0.08
    min_height_ratio: float = 0.01
    max_height_ratio: float = 0.08
    roi_margin_ratio: float = 0.01

    # Tray layout
    n_rows: int = 10
    n_cols: int = 5
    expected_seed_count: int = 50

    # Reference square
    reference_square_side_cm: float = 2.0

    # Individual seed crop size
    seed_crop_size_cm: float = 2.0


NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

CONFIG = ImageProcessingConfig(
    project_root=PROJECT_ROOT,
    original_images_dir=PROJECT_ROOT / "data" / "raw",
    rectified_images_dir=PROJECT_ROOT / "outputs" / "image_processing" / "rectified_tray_images",
    annotated_images_dir=PROJECT_ROOT / "outputs" / "image_processing" / "annotated_tray_images",
    individual_seeds_dir=PROJECT_ROOT / "outputs" / "image_processing" / "individual_seed_images",
    black_background_dir=PROJECT_ROOT / "outputs" / "image_processing" / "germination_black_background",
    processed_dataset_path=PROJECT_ROOT / "data" / "processed" / "image_measurements.csv",
    germination_metadata_path=PROJECT_ROOT / "data" / "processed" / "analytical_dataset.csv",
)

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
KERNEL = np.ones((CONFIG.kernel_size, CONFIG.kernel_size), np.uint8)
LOWER_BLUE = np.array(CONFIG.lower_blue, dtype=np.uint8)
UPPER_BLUE = np.array(CONFIG.upper_blue, dtype=np.uint8)

CONFIG

ImageProcessingConfig(project_root=PosixPath('/Users/claudemir/Insync/claudemirmota@gmail.com/Google Drive/GitHub/Sem Título'), original_images_dir=PosixPath('/Users/claudemir/Insync/claudemirmota@gmail.com/Google Drive/GitHub/Sem Título/data/raw'), rectified_images_dir=PosixPath('/Users/claudemir/Insync/claudemirmota@gmail.com/Google Drive/GitHub/Sem Título/outputs/image_processing/rectified_tray_images'), annotated_images_dir=PosixPath('/Users/claudemir/Insync/claudemirmota@gmail.com/Google Drive/GitHub/Sem Título/outputs/image_processing/annotated_tray_images'), individual_seeds_dir=PosixPath('/Users/claudemir/Insync/claudemirmota@gmail.com/Google Drive/GitHub/Sem Título/outputs/image_processing/individual_seed_images'), black_background_dir=PosixPath('/Users/claudemir/Insync/claudemirmota@gmail.com/Google Drive/GitHub/Sem Título/outputs/image_processing/germination_black_background'), processed_dataset_path=PosixPath('/Users/claudemir/Insync/claudemirmota@gmail.com/Google Dri

## 2. General Utilities

In [2]:
def ensure_directories(config: ImageProcessingConfig = CONFIG) -> None:
    """Create output directories used by the pipeline."""
    for path in [
        config.rectified_images_dir,
        config.annotated_images_dir,
        config.individual_seeds_dir,
        config.black_background_dir,
        config.processed_dataset_path.parent,
    ]:
        path.mkdir(parents=True, exist_ok=True)


def list_images(directory: Path, extensions: tuple[str, ...] = IMAGE_EXTENSIONS) -> list[Path]:
    """Return image files in a directory, sorted by filename."""
    if not directory.exists():
        return []
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in extensions)


def load_image(path: Path) -> np.ndarray:
    """Load an image with OpenCV and fail with a clear message if unavailable."""
    image = cv2.imread(str(path))
    if image is None:
        raise OSError(f"Could not load image: {path}")
    return image


def save_image(path: Path, image: np.ndarray) -> None:
    """Save an image and create its parent directory if needed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    ok = cv2.imwrite(str(path), image)
    if not ok:
        raise OSError(f"Could not save image: {path}")


def show_bgr_image(image: np.ndarray, title: str | None = None, figsize: tuple[int, int] = (8, 6)) -> None:
    """Display an OpenCV BGR image in notebook RGB format."""
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


def preview_saved_image(path: Path, width: int = 900) -> None:
    """Display an image file if it exists."""
    if path.exists():
        display(Markdown(f"### `{path.name}`"))
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"Image not found: `{path}`"))

## 3. EVA Detection and Perspective Correction

This step detects the blue EVA region and applies a perspective transform so that each tray image behaves like a scanner-aligned image.

In [3]:
def create_blue_mask(image: np.ndarray) -> np.ndarray:
    """Create a binary mask for the blue EVA background."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER_BLUE, UPPER_BLUE)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, KERNEL)
    return mask


def find_largest_contour(mask: np.ndarray) -> np.ndarray:
    """Find the largest external contour in a binary mask."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError("No contour found in mask.")
    return max(contours, key=cv2.contourArea)


def approximate_to_quadrilateral(contour: np.ndarray) -> np.ndarray:
    """Approximate a contour to four points, falling back to minAreaRect."""
    epsilon = 0.02 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    if len(approx) != 4:
        approx = cv2.boxPoints(cv2.minAreaRect(contour)).astype(np.intp)
    return approx


def order_points(points: np.ndarray) -> np.ndarray:
    """Order four points as top-left, top-right, bottom-right, bottom-left."""
    pts = points.reshape(4, 2)
    rect = np.zeros((4, 2), dtype="float32")

    point_sum = pts.sum(axis=1)
    rect[0] = pts[np.argmin(point_sum)]
    rect[2] = pts[np.argmax(point_sum)]

    point_diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(point_diff)]
    rect[3] = pts[np.argmax(point_diff)]
    return rect


def perspective_transform(image: np.ndarray, corners: np.ndarray) -> np.ndarray:
    """Apply perspective correction from ordered quadrilateral corners."""
    width_a = np.linalg.norm(corners[2] - corners[3])
    width_b = np.linalg.norm(corners[1] - corners[0])
    height_a = np.linalg.norm(corners[1] - corners[2])
    height_b = np.linalg.norm(corners[0] - corners[3])

    width = int(max(width_a, width_b))
    height = int(max(height_a, height_b))
    destination = np.array(
        [[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]],
        dtype="float32",
    )
    matrix = cv2.getPerspectiveTransform(corners, destination)
    return cv2.warpPerspective(image, matrix, (width, height))


def rectify_tray_image(image: np.ndarray) -> np.ndarray:
    """Crop the EVA area and correct perspective."""
    mask = create_blue_mask(image)
    contour = find_largest_contour(mask)
    quadrilateral = approximate_to_quadrilateral(contour)
    corners = order_points(quadrilateral)
    return perspective_transform(image, corners)


def rectify_image_directory(config: ImageProcessingConfig = CONFIG) -> pd.DataFrame:
    """Rectify all raw tray images and save them to the configured output directory."""
    ensure_directories(config)
    rows = []
    image_paths = list_images(config.original_images_dir)
    if not image_paths:
        raise FileNotFoundError(f"No images found in {config.original_images_dir}")

    for image_path in image_paths:
        try:
            rectified = rectify_tray_image(load_image(image_path))
            output_path = config.rectified_images_dir / image_path.name
            save_image(output_path, rectified)
            rows.append({"image": image_path.name, "status": "ok", "output_path": str(output_path)})
        except Exception as exc:
            rows.append({"image": image_path.name, "status": "error", "message": str(exc)})
    return pd.DataFrame(rows)

## 4. Calibration Square Detection

The white reference square is used to estimate pixels per centimeter. The default side length is 2 cm.

In [4]:
def detect_reference_square(
    image: np.ndarray,
    side_cm: float = CONFIG.reference_square_side_cm,
) -> tuple[float, np.ndarray]:
    """Detect the white reference square and return pixels per centimeter plus an annotated image."""
    output = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, KERNEL)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, KERNEL)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best_contour = None
    best_area = 0.0

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < 100 or area > 1_600_000:
            continue
        perimeter = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.05 * perimeter, True)
        if not 4 <= len(approx) <= 10:
            continue

        points = approx.reshape(-1, 2)
        sides = [np.linalg.norm(points[i] - points[(i + 1) % len(points)]) for i in range(len(points))]
        if 50 <= max(sides) <= 400 and area > best_area:
            best_area = area
            best_contour = approx

    if best_contour is None:
        raise RuntimeError("Reference square not found.")

    x, y, width, height = cv2.boundingRect(best_contour)
    px_per_cm_width = width / side_cm
    px_per_cm_height = height / side_cm
    relative_difference = abs(px_per_cm_width - px_per_cm_height) / max(px_per_cm_width, px_per_cm_height)
    px_per_cm = max(px_per_cm_width, px_per_cm_height) if relative_difference > 0.2 else (px_per_cm_width + px_per_cm_height) / 2

    cv2.polylines(output, [best_contour], isClosed=True, color=(0, 255, 0), thickness=5)
    cv2.putText(
        output,
        f"{px_per_cm:.2f} px/cm",
        (x, max(25, y - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 0),
        3,
    )
    return px_per_cm, output

## 5. Seed Segmentation, Filtering, and Grid Labeling

In [5]:
def remove_blue_background(image: np.ndarray) -> np.ndarray:
    """Suppress blue EVA pixels from the image."""
    mask = create_blue_mask(image)
    mask_inv = cv2.bitwise_not(mask)
    return cv2.bitwise_and(image, image, mask=mask_inv)


def segment_seeds(image: np.ndarray) -> np.ndarray:
    """Segment seeds using Otsu thresholding combined with edge information."""
    no_blue = remove_blue_background(image)
    gray = cv2.cvtColor(no_blue, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    _, mask_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    edges = cv2.Canny(gray, 40, 120)
    edges = cv2.dilate(edges, KERNEL, iterations=1)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, KERNEL)

    combined = cv2.bitwise_or(mask_otsu, edges)
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, KERNEL)
    combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN, KERNEL)
    return combined


def circularity(contour: np.ndarray) -> float:
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    return 0.0 if perimeter == 0 else 4 * np.pi * area / (perimeter**2)


def centroid(contour: np.ndarray) -> tuple[float, float] | None:
    moments = cv2.moments(contour)
    if moments["m00"] == 0:
        return None
    return moments["m10"] / moments["m00"], moments["m01"] / moments["m00"]


def contour_inside_eva_roi(contour: np.ndarray, eva_contour: np.ndarray, margin_ratio: float) -> bool:
    center = centroid(contour)
    if center is None:
        return False
    cx, cy = center
    x, y, width, height = cv2.boundingRect(eva_contour)
    margin_x = int(width * margin_ratio)
    margin_y = int(height * margin_ratio)
    return (x + margin_x <= cx <= x + width - margin_x) and (y + margin_y <= cy <= y + height - margin_y)


def find_valid_seed_contours(mask: np.ndarray, image_shape: tuple[int, int, int], eva_contour: np.ndarray) -> list[np.ndarray]:
    """Filter seed contours using geometry and position inside the EVA ROI."""
    height, width = image_shape[:2]
    image_area = height * width
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    valid = []
    for contour in contours:
        hull = cv2.convexHull(contour)
        area = cv2.contourArea(hull)
        x, y, w, h = cv2.boundingRect(hull)
        if (
            CONFIG.min_area_ratio < area / image_area < CONFIG.max_area_ratio
            and CONFIG.min_width_ratio < w / width < CONFIG.max_width_ratio
            and CONFIG.min_height_ratio < h / height < CONFIG.max_height_ratio
            and circularity(hull) > 0.35
            and contour_inside_eva_roi(hull, eva_contour, CONFIG.roi_margin_ratio)
        ):
            valid.append(hull)
    return valid


def compute_centroids(contours: Iterable[np.ndarray]) -> list[tuple[float, float, np.ndarray]]:
    centers = []
    for contour in contours:
        center = centroid(contour)
        if center is not None:
            centers.append((center[0], center[1], contour))
    return centers


def cluster_1d(values: list[float], n_clusters: int) -> list[int]:
    if len(values) < n_clusters:
        raise ValueError(f"Cannot assign {n_clusters} clusters from only {len(values)} values.")
    values_array = np.array(values).reshape(-1, 1)
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init="auto").fit(values_array)
    centers = sorted([(index, center[0]) for index, center in enumerate(kmeans.cluster_centers_)], key=lambda item: item[1])
    remap = {old: new + 1 for new, (old, _) in enumerate(centers)}
    return [remap[label] for label in kmeans.labels_]


def assign_grid_labels(centroids: list[tuple[float, float, np.ndarray]]) -> list[dict]:
    """Assign seed labels based on row/column clustering."""
    row_labels = cluster_1d([item[1] for item in centroids], CONFIG.n_rows)
    column_labels = cluster_1d([item[0] for item in centroids], CONFIG.n_cols)

    labeled = []
    for (cx, cy, contour), row, column in zip(centroids, row_labels, column_labels):
        x, y, width, height = cv2.boundingRect(contour)
        labeled.append(
            {
                "grid_row": row,
                "grid_column": column,
                "seed_image": f"s_{row}{column}.png",
                "centroid_x_px": cx,
                "centroid_y_px": cy,
                "area_px": cv2.contourArea(contour),
                "perimeter_px": cv2.arcLength(contour, True),
                "width_px": width,
                "height_px": height,
                "contour": contour,
            }
        )
    return sorted(labeled, key=lambda item: (item["grid_row"], item["grid_column"]))


## 6. Analyze One Tray Image

In [6]:
def analyze_tray_image(image: np.ndarray) -> tuple[np.ndarray, pd.DataFrame, list[dict]]:
    """Detect, measure, label, and annotate seeds in one tray image."""
    blue_mask = create_blue_mask(image)
    eva_contour = find_largest_contour(blue_mask)
    seed_mask = segment_seeds(image)
    valid_contours = find_valid_seed_contours(seed_mask, image.shape, eva_contour)
    labeled_seeds = assign_grid_labels(compute_centroids(valid_contours))
    px_per_cm, annotated = detect_reference_square(image)

    x, y, width, height = cv2.boundingRect(eva_contour)
    margin_x = int(width * CONFIG.roi_margin_ratio)
    margin_y = int(height * CONFIG.roi_margin_ratio)
    cv2.rectangle(annotated, (x + margin_x, y + margin_y), (x + width - margin_x, y + height - margin_y), (255, 0, 0), 2)

    rows = []
    for seed in labeled_seeds:
        contour = seed["contour"]
        cx, cy = int(seed["centroid_x_px"]), int(seed["centroid_y_px"])
        cv2.drawContours(annotated, [contour], -1, (0, 255, 0), 2)
        cv2.circle(annotated, (cx, cy), 3, (0, 0, 255), -1)
        cv2.putText(
            annotated,
            seed["seed_image"].replace(".png", ""),
            (cx - 25, cy - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1,
            cv2.LINE_AA,
        )
        rows.append({key: value for key, value in seed.items() if key != "contour"} | {"px_per_cm": px_per_cm})

    measurements = pd.DataFrame(rows).sort_values(["grid_row", "grid_column"]).reset_index(drop=True)
    if len(measurements) != CONFIG.expected_seed_count:
        print(f"Warning: expected {CONFIG.expected_seed_count} seeds, found {len(measurements)}.")
    return annotated, measurements, labeled_seeds


def parse_lot_from_image_name(path: Path) -> str:
    match = re.match(r"L(\d+)", path.stem)
    return match.group(1) if match else path.stem


def extract_lot_sheet_from_filename(path: Path) -> tuple[str, str]:
    match = re.match(r"(L\d+)(G\d+)", path.stem)
    if not match:
        raise ValueError(f"Invalid image name. Expected LxGy format, got: {path.name}")
    return match.group(1), match.group(2)


## 7. Batch Processing and Measurement Export

In [7]:
ANALYSIS_COLUMN_ORDER = [
    "lot",
    "image_id",
    "seed_image",
    "seed_id",
    "grid_row",
    "grid_column",
    "calibration_px_per_cm",
    "area_px",
    "perimeter_px",
    "width_px",
    "height_px",
    "centroid_x_px",
    "centroid_y_px",
    "area_cm2",
    "area_mm2",
    "perimeter_cm",
    "perimeter_mm",
    "width_cm",
    "height_cm",
    "length_mm",
    "width_mm",
    "germination_observed",
    "root_length_mm",
    "germinated_root_gt_35mm",
]


def parse_numeric_series(series: pd.Series) -> pd.Series:
    """Convert numeric text robustly, including comma decimal separators."""
    return pd.to_numeric(series.astype(str).str.replace(",", ".", regex=False), errors="coerce")


def add_physical_units(df: pd.DataFrame) -> pd.DataFrame:
    """Add physical measurements from pixel measurements using px/cm calibration.

    The output names follow the analysis notebook contract. In the original dataset,
    `length_mm` corresponds to the horizontal bounding-box dimension and `width_mm`
    corresponds to the vertical bounding-box dimension.
    """
    converted = df.copy()
    px_per_cm = converted["calibration_px_per_cm"]
    converted["area_cm2"] = converted["area_px"] / (px_per_cm**2)
    converted["area_mm2"] = converted["area_cm2"] * 100
    converted["perimeter_cm"] = converted["perimeter_px"] / px_per_cm
    converted["perimeter_mm"] = converted["perimeter_cm"] * 10
    converted["width_cm"] = converted["width_px"] / px_per_cm
    converted["height_cm"] = converted["height_px"] / px_per_cm
    converted["length_mm"] = converted["width_cm"] * 10
    converted["width_mm"] = converted["height_cm"] * 10
    return converted


def load_germination_metadata(metadata_path: Path | None) -> pd.DataFrame:
    """Load germination metadata keyed by seed_id."""
    if metadata_path is None or not metadata_path.exists():
        return pd.DataFrame(columns=["seed_id", "germination_observed", "root_length_mm"])

    metadata = pd.read_csv(metadata_path)
    required = ["seed_id", "germination_observed", "root_length_mm"]
    missing = [column for column in required if column not in metadata.columns]
    if missing:
        raise ValueError(f"Germination metadata is missing required columns: {missing}")

    metadata = metadata[required].copy()
    metadata["germination_observed"] = pd.to_numeric(metadata["germination_observed"], errors="coerce").astype("Int64")
    metadata["root_length_mm"] = parse_numeric_series(metadata["root_length_mm"])
    return metadata.drop_duplicates(subset="seed_id", keep="first")


def add_germination_columns(measurements: pd.DataFrame, config: ImageProcessingConfig = CONFIG) -> pd.DataFrame:
    """Merge germination metadata and derive the threshold-based target."""
    metadata = load_germination_metadata(config.germination_metadata_path)
    merged = measurements.merge(metadata, on="seed_id", how="left")
    merged["germination_observed"] = merged["germination_observed"].astype("Int64")
    merged["germinated_root_gt_35mm"] = (merged["root_length_mm"] > 35.0).astype("Int64")
    return merged


def process_rectified_images(config: ImageProcessingConfig = CONFIG) -> pd.DataFrame:
    """Analyze all rectified tray images and export measurements using analysis-ready headers."""
    ensure_directories(config)
    image_paths = list_images(config.rectified_images_dir)
    if not image_paths:
        raise FileNotFoundError(f"No rectified images found in {config.rectified_images_dir}")

    tables = []
    for image_path in image_paths:
        print(f"Analyzing {image_path.name}")
        image = load_image(image_path)
        annotated, measurements, _ = analyze_tray_image(image)
        measurements.insert(0, "image_id", image_path.stem)
        measurements.insert(0, "lot", parse_lot_from_image_name(image_path))
        measurements["seed_id"] = measurements["image_id"] + "_" + measurements["seed_image"]
        tables.append(measurements)
        save_image(config.annotated_images_dir / f"{image_path.stem}_annotated.jpg", annotated)

    dataset = pd.concat(tables, ignore_index=True)
    dataset = dataset.rename(columns={"px_per_cm": "calibration_px_per_cm"})
    measurement_columns = [
        "lot",
        "image_id",
        "seed_image",
        "seed_id",
        "grid_row",
        "grid_column",
        "calibration_px_per_cm",
        "area_px",
        "perimeter_px",
        "width_px",
        "height_px",
        "centroid_x_px",
        "centroid_y_px",
    ]
    dataset = add_physical_units(dataset[measurement_columns])
    dataset = add_germination_columns(dataset, config)

    numeric_lot = pd.to_numeric(dataset["lot"], errors="coerce")
    if numeric_lot.notna().all():
        dataset["lot"] = numeric_lot.astype(int)
    else:
        dataset["lot"] = dataset["lot"].astype(str)

    dataset = dataset.sort_values(["lot", "image_id", "grid_row", "grid_column"]).reset_index(drop=True)
    dataset = dataset[ANALYSIS_COLUMN_ORDER]
    dataset.to_csv(config.processed_dataset_path, index=False)
    return dataset


## 8. Optional Germination Metadata Merge

Use this section only if you have a legacy germination CSV with seed identifiers that can be matched to `seed_id`.

In [8]:
# Germination metadata is merged automatically in process_rectified_images().
# The metadata CSV is configured with CONFIG.germination_metadata_path and must contain:
# seed_id, germination_observed, root_length_mm
#
# The exported CSV at CONFIG.processed_dataset_path is analysis-ready and follows
# the same column contract expected by analysis_pipeline.ipynb.


## 9. Crop Individual Seed Images

In [9]:
def crop_seed(image: np.ndarray, cx: float, cy: float, px_per_cm: float, size_cm: float = CONFIG.seed_crop_size_cm) -> np.ndarray:
    """Crop a square region centered on the seed centroid."""
    half_size_px = int((size_cm / 2.0) * px_per_cm * 0.5)
    x1 = int(cx - half_size_px)
    y1 = int(cy - half_size_px)
    x2 = int(cx + half_size_px)
    y2 = int(cy + half_size_px)
    height, width = image.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(width, x2), min(height, y2)
    return image[y1:y2, x1:x2]


def save_individual_seeds(
    image: np.ndarray,
    labeled_seeds: list[dict],
    px_per_cm: float,
    output_dir: Path,
    lot: str,
    sheet: str,
) -> None:
    """Save individual seed crops for one tray image."""
    output_dir.mkdir(parents=True, exist_ok=True)
    for seed in labeled_seeds:
        crop = crop_seed(image, seed["centroid_x_px"], seed["centroid_y_px"], px_per_cm)
        filename = f"{lot}{sheet}_{seed['seed_image']}"
        save_image(output_dir / filename, crop)


def crop_all_individual_seeds(config: ImageProcessingConfig = CONFIG) -> pd.DataFrame:
    """Crop individual seed images from all rectified tray images."""
    ensure_directories(config)
    rows = []
    for image_path in list_images(config.rectified_images_dir):
        lot, sheet = extract_lot_sheet_from_filename(image_path)
        image = load_image(image_path)
        blue_mask = create_blue_mask(image)
        eva_contour = find_largest_contour(blue_mask)
        seed_mask = segment_seeds(image)
        valid_contours = find_valid_seed_contours(seed_mask, image.shape, eva_contour)
        labeled_seeds = assign_grid_labels(compute_centroids(valid_contours))
        px_per_cm, _ = detect_reference_square(image)
        lot_dir = config.individual_seeds_dir / lot
        save_individual_seeds(image, labeled_seeds, px_per_cm, lot_dir, lot, sheet)
        rows.append({"image": image_path.name, "lot": lot, "sheet": sheet, "seeds_saved": len(labeled_seeds), "output_dir": str(lot_dir)})
    return pd.DataFrame(rows)


## 10. Optional Seed Class Folder Split

In [10]:
def split_seed_images_by_germination(
    dataset: pd.DataFrame,
    source_root: Path,
    germinated_dir: Path,
    non_germinated_dir: Path,
    germination_column: str = "germination_observed",
) -> pd.DataFrame:
    """Copy individual seed images into germinated and non-germinated folders."""
    germinated_dir.mkdir(parents=True, exist_ok=True)
    non_germinated_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for row in dataset.itertuples(index=False):
        seed_id = getattr(row, "seed_id")
        germination_value = getattr(row, germination_column)
        lot_match = re.match(r"(L\d+)", seed_id)
        if lot_match is None:
            rows.append({"seed_id": seed_id, "status": "invalid_lot"})
            continue

        source_path = source_root / lot_match.group(1) / seed_id
        destination = germinated_dir / seed_id if germination_value == 1 else non_germinated_dir / seed_id
        if source_path.exists():
            shutil.copy2(source_path, destination)
            rows.append({"seed_id": seed_id, "status": "copied", "destination": str(destination)})
        else:
            rows.append({"seed_id": seed_id, "status": "missing_source", "source": str(source_path)})
    return pd.DataFrame(rows)


## 11. Convert Blue EVA Background to Black

In [11]:
def remove_blue_to_black(image: np.ndarray) -> np.ndarray:
    """Convert blue EVA pixels to pure black while preserving seed pixels."""
    mask_blue = create_blue_mask(image)
    mask_seed = cv2.bitwise_not(mask_blue)
    result = np.zeros_like(image)
    result[mask_seed > 0] = image[mask_seed > 0]
    return result


def process_seed_folder_to_black_background(input_root: Path, output_root: Path) -> pd.DataFrame:
    """Process all seed PNGs under lot subfolders and preserve folder structure."""
    output_root.mkdir(parents=True, exist_ok=True)
    rows = []
    for lot_dir in sorted(path for path in input_root.iterdir() if path.is_dir()):
        output_lot_dir = output_root / lot_dir.name
        output_lot_dir.mkdir(parents=True, exist_ok=True)
        for image_path in sorted(lot_dir.glob("*.png")):
            image = load_image(image_path)
            processed = remove_blue_to_black(image)
            output_path = output_lot_dir / image_path.name
            save_image(output_path, processed)
            rows.append({"lot": lot_dir.name, "image": image_path.name, "output_path": str(output_path)})
    return pd.DataFrame(rows)

## 12. Pipeline Execution Cells

Run only the steps you need. These cells are intentionally separated so that large image processing jobs do not start accidentally when the notebook is opened on GitHub.

In [12]:
# Step 1: Crop EVA and correct perspective from raw images.
rectification_log = rectify_image_directory(CONFIG)
rectification_log

,image,status,output_path
0,L10G1.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...
1,L10G10.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...
2,L10G11.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...
3,L10G12.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...
4,L10G13.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...
...,...,...,...
315,L9G5.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...
316,L9G6.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...
317,L9G7.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...
318,L9G8.jpg,ok,/Users/claudemir/Insync/claudemirmota@gmail.co...


In [13]:
# Step 2: Analyze rectified images and export the measurement CSV.
dataset = process_rectified_images(CONFIG)
dataset.head()

Analyzing L10G1.jpg
Analyzing L10G10.jpg
Analyzing L10G11.jpg
Analyzing L10G12.jpg
Analyzing L10G13.jpg
Analyzing L10G14.jpg
Analyzing L10G15.jpg
Analyzing L10G16.jpg
Analyzing L10G17.jpg
Analyzing L10G18.jpg
Analyzing L10G19.jpg
Analyzing L10G2.jpg
Analyzing L10G20.jpg
Analyzing L10G3.jpg
Analyzing L10G4.jpg
Analyzing L10G5.jpg
Analyzing L10G6.jpg
Analyzing L10G7.jpg
Analyzing L10G8.jpg
Analyzing L10G9.jpg
Analyzing L11G1.jpg
Analyzing L11G10.jpg
Analyzing L11G11.jpg
Analyzing L11G12.jpg
Analyzing L11G13.jpg
Analyzing L11G14.jpg
Analyzing L11G15.jpg
Analyzing L11G16.jpg
Analyzing L11G17.jpg
Analyzing L11G18.jpg
Analyzing L11G19.jpg
Analyzing L11G2.jpg
Analyzing L11G20.jpg
Analyzing L11G3.jpg
Analyzing L11G4.jpg
Analyzing L11G5.jpg
Analyzing L11G6.jpg
Analyzing L11G7.jpg
Analyzing L11G8.jpg
Analyzing L11G9.jpg
Analyzing L12G1.jpg
Analyzing L12G10.jpg
Analyzing L12G11.jpg
Analyzing L12G12.jpg
Analyzing L12G13.jpg
Analyzing L12G14.jpg
Analyzing L12G15.jpg
Analyzing L12G16.jpg
Analyzing L

,lot,image_id,seed_image,seed_id,grid_row,grid_column,calibration_px_per_cm,area_px,perimeter_px,width_px,...,area_mm2,perimeter_cm,perimeter_mm,width_cm,height_cm,length_mm,width_mm,germination_observed,root_length_mm,germinated_root_gt_35mm
0,1,L1G1,s_11.png,L1G1_s_11.png,1,1,183.0,2178.5,173.018163,46,...,6.505121,0.945454,9.454544,0.251366,0.344262,2.513661,3.442623,1,70.9,1
1,1,L1G1,s_12.png,L1G1_s_12.png,1,2,183.0,2500.5,181.120181,58,...,7.466631,0.989728,9.897278,0.316940,0.289617,3.169399,2.896175,1,42.9,1
2,1,L1G1,s_13.png,L1G1_s_13.png,1,3,183.0,2613.0,183.699487,54,...,7.802562,1.003822,10.038223,0.295082,0.333333,2.950820,3.333333,1,34.3,0
3,1,L1G1,s_14.png,L1G1_s_14.png,1,4,183.0,2840.0,199.942610,67,...,8.480397,1.092583,10.925826,0.366120,0.322404,3.661202,3.224044,1,61.9,1
4,1,L1G1,s_15.png,L1G1_s_15.png,1,5,183.0,3010.5,199.452959,67,...,8.989519,1.089907,10.899069,0.366120,0.322404,3.661202,3.224044,1,40.9,1


In [14]:
# Step 3: Crop individual seed images.
# crop_log = crop_all_individual_seeds(CONFIG)
# crop_log.head()

In [15]:
# Step 4: Convert blue background in individual seed images to black.
# black_background_log = process_seed_folder_to_black_background(
#     CONFIG.individual_seeds_dir,
#     CONFIG.black_background_dir,
# )
# black_background_log.head()

## 13. Quick Visual Sanity Check

After running one of the processing steps, set `preview_path` to an output image and run the cell below.

In [16]:
# preview_path = CONFIG.annotated_images_dir / "L1G1_annotated.jpg"
# preview_saved_image(preview_path, width=900)